In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [2]:
class LogisticRegression:
    def __init__(self):
        self.optimizer = None
        self.hat_m_b = None
        self.hat_m_w = None
        self.r_v_b = None
        self.r_v_w = None
        self.hat_v_w = None
        self.hat_v_b = None
        self.x = None
        self.y = None
        self._y = None
        self.w = None
        self.b = None
        self.w_error = None
        self.b_error = None
        self.m_w = None
        self.m_b = None
        self.v_w = None
        self.v_b = None
        self.epoch_count = 1
        self.epoch_limit = None
    def predict(self,x):
        out = 1/(1 + np.exp(-(np.dot(x , self.w) + self.b)))
        return out
    def fit(self,x,y,epochs = 100,optimizer = 'gd'):
        self.x = x
        self.y = y
        self.w = np.zeros(x.shape[1])
        self.b = 0
        self._y = self.predict(self.x)
        self.epoch_limit = epochs
        self.optimizer = optimizer
        #initialized the momentum to avoid type error in first iteration
        self.v_w = np.zeros(self.x.shape[1])
        self.m_w = np.zeros(self.x.shape[1])
        self.m_b = 0
        self.v_b = 0
        #####
        self.b_error = -(np.mean(self.y - self._y))
        self.w_error = -(np.dot(self.x.transpose(),self.y - self._y))/self.x.shape[0]
        if self.optimizer == "adam":
            update_step = self.adam
        elif self.optimizer == "RMSprop":
            update_step = self.RMSprop
        elif self.optimizer == "momentum":
            update_step = self.momentum
        elif self.optimizer == "gd":
            update_step = self.gd
        else:
            raise ValueError(f"Unknown optimizer: {self.optimizer}")
        while self.epoch_count <= self.epoch_limit:
            self._y = self.predict(self.x)
            self.loss = sum([-t_hat*np.log(t) - (1-t_hat)*np.log(1-t) for t , t_hat in zip(self._y,self.y)])/len(self.y)
            self.b_error = -(np.mean(self.y - self._y))
            self.w_error = -(np.dot(self.x.transpose(),self.y - self._y))/self.x.shape[0]
            update_step()
            print(f'Epoch : {self.epoch_count} Loss : {self.loss}')
            self.epoch_count += 1
    def adam(self, beta1 = 0.9 , beta2 = 0.999 ,epsilon = 10**-8,gamma = 0.0003):
        #momentum variables
        self.m_w = beta1*self.m_w + (1 - beta1)*self.w_error
        self.m_b = beta1*self.m_b + (1 - beta1)*self.b_error
        #RMSpropFactors
        self.v_w = beta2*self.v_w + (1-beta2)*(self.w_error**2)
        self.v_b = beta2*self.v_b + (1-beta2)*(self.b_error**2)
        #bias correcting the variance values
        self.hat_v_w = self.v_w / (1 - beta2**self.epoch_count)
        self.hat_v_b = self.v_b / (1 - beta2**self.epoch_count)
        #RMSpropFactvectors for updating the final values
        self.r_v_w = 1 / (np.sqrt(self.hat_v_w + epsilon))
        self.r_v_b = 1 / (np.sqrt(self.hat_v_b + epsilon))
        #bias correcting the momentum values
        self.hat_m_w = self.m_w / (1 - beta1**self.epoch_count)
        self.hat_m_b = self.m_b / (1 - beta1**self.epoch_count)
        #Updating the final value
        self.b = self.b - self.hat_m_b*self.r_v_b*gamma
        self.w = self.w - np.multiply(self.hat_m_w,self.r_v_w)*gamma
    def RMSprop(self,beta2 = 0.999,epsilon = 10**-8, gamma = 0.001):
        self.v_w = beta2*self.v_w + (1-beta2)*(self.w_error**2)
        self.v_b = beta2*self.v_b + (1-beta2)*(self.b_error**2)
        self.r_v_w = 1 / (np.sqrt(self.v_w + epsilon))
        self.r_v_b = 1 / (np.sqrt(self.v_b + epsilon))
        self.b = self.b - self.b_error*self.r_v_b*gamma
        self.w = self.w - np.multiply(self.w_error,self.r_v_w)*gamma
    def momentum(self,beta1 = 0.9 , gamma = 0.001):
        self.m_w = beta1*self.m_w + (1 - beta1)*self.w_error
        self.m_b = beta1*self.m_b + (1 - beta1)*self.b_error
        self.hat_m_w = self.m_w / (1 - beta1**self.epoch_count)
        self.hat_m_b = self.m_b / (1 - beta1**self.epoch_count)
        self.b = self.b - self.hat_m_b*gamma
        self.w = self.w - self.hat_m_w*gamma
    def gd(self,gamma = 0.005):
        self.w = self.w - gamma*self.w_error
        self.b = self.b - self.b_error*gamma



In [3]:
df = pd.read_csv('Iris.csv')
training_df = df[df.Species == 'Iris-setosa']
x_variable = training_df.loc[:,['SepalLengthCm','SepalWidthCm']]
y_variable = training_df.loc[:,['PetalLengthCm']]
x_train,x_test,y_train,y_test = train_test_split(x_variable, y_variable, test_size = 0.10)

In [7]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform it
x_train_scaled = scaler.fit_transform(x_train.values)

# --- IMPORTANT ---
# Use that *same* scaler to transform your test data
x_test_scaled = scaler.transform(x_test.values)

# Now, train your model on the SCALED data
model = LogisticRegression()
model.fit(x_train_scaled, y_train.values.ravel(), epochs = 125,optimizer = "gd")
y_pred = model.predict(x_test_scaled)

# And predict on the SCALED test data
predictions = model.predict(x_test_scaled)
print(predictions)
#
# # Now plot your results
# plt.scatter(y_test, predictions)
# print(model.class_class())



Epoch : 1 Loss : 0.6931471805599445
Epoch : 2 Loss : 0.6885670692646365
Epoch : 3 Loss : 0.6839984316479186
Epoch : 4 Loss : 0.6794412387361306
Epoch : 5 Loss : 0.6748954614957
Epoch : 6 Loss : 0.6703610708348292
Epoch : 7 Loss : 0.6658380376051729
Epoch : 8 Loss : 0.6613263326035066
Epoch : 9 Loss : 0.6568259265733826
Epoch : 10 Loss : 0.6523367902067767
Epoch : 11 Loss : 0.6478588941457243
Epoch : 12 Loss : 0.6433922089839458
Epoch : 13 Loss : 0.6389367052684602
Epoch : 14 Loss : 0.6344923535011906
Epoch : 15 Loss : 0.6300591241405539
Epoch : 16 Loss : 0.6256369876030454
Epoch : 17 Loss : 0.6212259142648081
Epoch : 18 Loss : 0.6168258744631919
Epoch : 19 Loss : 0.6124368384983023
Epoch : 20 Loss : 0.6080587766345371
Epoch : 21 Loss : 0.6036916591021123
Epoch : 22 Loss : 0.5993354560985756
Epoch : 23 Loss : 0.5949901377903094
Epoch : 24 Loss : 0.590655674314022
Epoch : 25 Loss : 0.5863320357782263
Epoch : 26 Loss : 0.582019192264709
Epoch : 27 Loss : 0.5777171138299855
Epoch : 28 Loss

In [23]:
class MulticlassClassification:
    def __init__(self):
        self.models = []

    def fit(self, X, y,epochs = 100):
        for y_i in np.unique(y):
            x_true = X[y == y_i]
            x_false = X[y != y_i]
            x_true_false = np.vstack((x_true, x_false))
            y_true = np.ones(x_true.shape[0])
            y_false = np.zeros(x_false.shape[0])
            y_true_false = np.hstack((y_true, y_false))
            model = LogisticRegression()
            model.fit(x_true_false, y_true_false,epochs = epochs)
            self.models.append([y_i, model])
    def predict(self, X):
        y_pred = [[label, model.predict(X)] for label, model in self.models]

        output = []

        for i in range(X.shape[0]):
            max_label = None
            max_prob = -10**5
            for j in range(len(y_pred)):
                prob = y_pred[j][1][i]
                if prob > max_prob:
                    max_label = y_pred[j][0]
                    max_prob = prob
            output.append(max_label)

        return output

In [13]:
xdf = pd.read_csv('Iris.csv')

In [14]:
ys = xdf.Species

In [15]:
xdf = xdf.drop('Species',axis = 1).to_numpy()

In [16]:
x_train,x_test,y_train,y_test = train_test_split(xdf, ys,stratify = ys, test_size = 0.10)

In [18]:
scaler = StandardScaler()

In [20]:
x_train = scaler.fit_transform(x_train)

In [21]:
x_test = scaler.transform(x_test)

In [25]:
Model = MulticlassClassification()
Model.fit(x_train, y_train,epochs = 200)
pd.Series(Model.predict(x_test) == y_test).value_counts()

Epoch : 1 Loss : 0.6931471805599462
Epoch : 2 Loss : 0.6894765300036197
Epoch : 3 Loss : 0.6858388154290087
Epoch : 4 Loss : 0.682233731121831
Epoch : 5 Loss : 0.6786609726698088
Epoch : 6 Loss : 0.6751202370167324
Epoch : 7 Loss : 0.6716112225144071
Epoch : 8 Loss : 0.6681336289725316
Epoch : 9 Loss : 0.6646871577065216
Epoch : 10 Loss : 0.661271511583326
Epoch : 11 Loss : 0.6578863950652434
Epoch : 12 Loss : 0.6545315142517966
Epoch : 13 Loss : 0.651206576919677
Epoch : 14 Loss : 0.6479112925608069
Epoch : 15 Loss : 0.6446453724185467
Epoch : 16 Loss : 0.6414085295220867
Epoch : 17 Loss : 0.6382004787190524
Epoch : 18 Loss : 0.6350209367063706
Epoch : 19 Loss : 0.6318696220594201
Epoch : 20 Loss : 0.628746255259516
Epoch : 21 Loss : 0.6256505587197538
Epoch : 22 Loss : 0.6225822568092576
Epoch : 23 Loss : 0.6195410758758695
Epoch : 24 Loss : 0.6165267442673162
Epoch : 25 Loss : 0.613538992350888
Epoch : 26 Loss : 0.6105775525316725
Epoch : 27 Loss : 0.6076421592693758
Epoch : 28 Loss

Species
True     14
False     1
Name: count, dtype: int64

In [26]:
print(Model.predict(x_test))

['Iris-setosa', 'Iris-virginica', 'Iris-virginica', 'Iris-versicolor', 'Iris-setosa', 'Iris-setosa', 'Iris-versicolor', 'Iris-versicolor', 'Iris-setosa', 'Iris-virginica', 'Iris-virginica', 'Iris-virginica', 'Iris-versicolor', 'Iris-virginica', 'Iris-setosa']
